In [1]:
%load_ext autoreload
%autoreload 2
import sys, json, time, gc
from pathlib import Path
import numpy as np
import polars as pl
import joblib

ROOT = Path.cwd().resolve()
while not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Không tìm thấy thư mục gốc chứa src/ và configs/')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.features.prepare import META
try:
    from src.features.prepare import AUDIT_COLS
except ImportError:
    AUDIT_COLS = ['person_1__app__sex_738L']

cfg = load_config()
PROC = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
SEX = 'person_1__app__sex_738L'

# ---- Danh sách artifact ----
for d in (MODELS, REPORTS):
    print(f'\n{d.name}/')
    for p in sorted(d.glob('champion_*')):
        print(f'  {p.name:45s} {p.stat().st_size/1e6:8.2f} MB')

# ---- Mô tả đối tượng đã lưu ----
def describe(obj, name):
    print(f'\n--- {name}: {type(obj)}')
    if isinstance(obj, dict):
        for k, v in obj.items():
            n = len(v) if hasattr(v, '__len__') and not isinstance(v, str) else v
            print(f'  {k}: {type(v).__name__} | {n}')
        return
    for a in ['feature_names_in_', 'n_features_in_', 'coef_', 'intercept_', 'C', 'penalty',
              'solver', 'class_weight', 'max_iter', 'variable_names', 'min_bin_size', 'binning_fit_params']:
        if hasattr(obj, a):
            v = getattr(obj, a)
            if hasattr(v, 'shape'):
                print(f'  {a}: shape {v.shape}')
            elif isinstance(v, (list, tuple)):
                print(f'  {a}: list[{len(v)}]')
            else:
                print(f'  {a}: {v}')

binB = joblib.load(MODELS / 'champion_binning_B.joblib')
logit62 = joblib.load(MODELS / 'champion_logit_B_full62.joblib')
describe(binB, 'champion_binning_B')
describe(logit62, 'champion_logit_B_full62')

# ---- 61 feature có còn trong model_table hiện tại? ----
feat62 = list(getattr(logit62, 'feature_names_in_', []))
TABLE_COLS = set(pl.scan_parquet(PROC / 'model_table.parquet').collect_schema().names())
print(f'\nfeat62: {len(feat62)} | có giới tính: {SEX in feat62}')
feat61 = [f for f in feat62 if f != SEX]
missing = [f for f in feat61 if f not in TABLE_COLS]
print(f'feat61: {len(feat61)} | không có trong model_table hiện tại: {len(missing)}')
print(missing)

# ---- Nội dung các report json của Champion ----
for p in sorted(REPORTS.glob('champion_*.json')):
    with open(p, encoding='utf-8') as f:
        obj = json.load(f)
    print(f'\n{p.name}:', list(obj.keys()) if isinstance(obj, dict) else type(obj).__name__)


models/
  champion_binning_A.joblib                         1.85 MB
  champion_binning_B.joblib                         2.02 MB
  champion_logit_A.joblib                           0.00 MB
  champion_logit_B.joblib                           0.00 MB
  champion_logit_B_full62.joblib                    0.00 MB

reports/
  champion_coef_A.csv                               0.00 MB
  champion_coef_B.csv                               0.00 MB
  champion_feature_lists.json                       0.05 MB
  champion_forward_path_B.csv                       0.00 MB
  champion_iv_A.csv                                 0.04 MB
  champion_iv_B.csv                                 0.05 MB
  champion_iv_compare.csv                           0.05 MB
  champion_score_bands_B.csv                        0.00 MB
  champion_scorecard_B.csv                          0.03 MB
  champion_scorecard_params.json                    0.00 MB
  champion_selected_after_corr.json                 0.01 MB
  champion_window_ab.

In [2]:
GENDER2 = 'person_1__app__gender_992L'
FEATS = [c for c in TABLE_COLS if c not in set(META) | set(AUDIT_COLS)]

# ---- Danh sách feature Champion ----
feat62 = list(logit62['feats'])
feat61 = [f for f in feat62 if f != SEX]
print(f'feat62: {len(feat62)} | có {SEX}: {SEX in feat62} | có {GENDER2}: {GENDER2 in feat62}')
print(f'feat61: {len(feat61)} | không có trong model_table: {[f for f in feat61 if f not in TABLE_COLS]}')
print(f'feat61 thiếu bộ chia bin: {[f for f in feat61 if f not in binB]}')
print(f'dropped_sign ({len(logit62["dropped_sign"])}):', logit62['dropped_sign'])

with open(REPORTS / 'champion_feature_lists.json', encoding='utf-8') as f:
    FL = json.load(f)
cf = FL['champion_feats']
print(f"\nchampion_feature_lists['champion_feats']: {len(cf)} | trùng feat62: {len(set(cf) & set(feat62))}")

# ---- gender_992L có trong 627 feature (Challenger) không? ----
print(f'\n{GENDER2} trong FEATS (627): {GENDER2 in FEATS} | trong model_table: {GENDER2 in TABLE_COLS}')

# ---- So sánh với sex_738L trên Train (tuần 0–43) ----
if GENDER2 in TABLE_COLS:
    g = (pl.scan_parquet(PROC / 'model_table.parquet')
           .filter(pl.col('WEEK_NUM') <= 43)
           .select([SEX, GENDER2, 'target'])
           .collect(engine='streaming'))
    print(f'Null: {SEX} {g[SEX].is_null().mean():.4f} | {GENDER2} {g[GENDER2].is_null().mean():.4f}')
    with pl.Config(tbl_rows=30):
        print(g.group_by([SEX, GENDER2]).agg(pl.len().alias('n'), pl.col('target').mean().round(4).alias('dr'))
               .sort('n', descending=True))
    del g; gc.collect()

feat62: 62 | có person_1__app__sex_738L: True | có person_1__app__gender_992L: False
feat61: 61 | không có trong model_table: []
feat61 thiếu bộ chia bin: []
dropped_sign (17): ['credit_bureau_a_1__lastupdate_388D__max', 'riskassesment_302T', 'secondquarter_766L', 'person_1__app__empl_employedtotal_800L', 'credit_bureau_a_1__instlamount_852A__mean', 'applprev_2__conts_type_509L__n_PHONE', 'applprev_1__inittransactioncode_279L__nuniq', 'applprev_1__credacc_status_367L__nuniq', 'maxdebt4_972A', 'applprev_1__employedfrom_700D__max', 'credit_bureau_a_1__numberofoverdueinstlmaxdat_148D__min', 'numinstpaidlate1d_3546852L', 'firstquarter_103L', 'pctinstlsallpaidearl3d_427L', 'avgdbddpdlast24m_3658932P', 'maxdpdlast24m_143P', 'pctinstlsallpaidlate6d_3546844L']

champion_feature_lists['champion_feats']: 2 | trùng feat62: 0

person_1__app__gender_992L trong FEATS (627): True | trong model_table: True
Null: person_1__app__sex_738L 0.0000 | person_1__app__gender_992L 0.9916
shape: (4, 4)
┌────────

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def gini_stability(week, y_true, y_score):
    week, y_true, y_score = map(np.asarray, (week, y_true, y_score))
    weeks = np.unique(week)
    ginis = np.array([2 * roc_auc_score(y_true[week == w], y_score[week == w]) - 1 for w in weeks])
    a, b = np.polyfit(weeks.astype(float), ginis, 1)
    residuals = ginis - (a * weeks + b)
    return {'gini_overall': float(2 * roc_auc_score(y_true, y_score) - 1),
            'mean_gini': float(ginis.mean()), 'slope_a': float(a), 'std_resid': float(residuals.std()),
            'stability': float(ginis.mean() + 88.0 * min(0.0, a) - 0.5 * residuals.std()),
            'weekly': dict(zip(weeks.tolist(), ginis.round(4).tolist()))}

print('Cột tên có sex/gender:', [c for c in sorted(TABLE_COLS) if 'sex' in c.lower() or 'gender' in c.lower()])

CH_WEEKS, VALID_WEEKS = (20, 43), (44, 51)
lf = pl.scan_parquet(PROC / 'model_table.parquet')
cols = ['case_id', 'WEEK_NUM', 'target'] + feat62
tr = lf.filter(pl.col('WEEK_NUM').is_between(*CH_WEEKS)).select(cols).collect(engine='streaming')
va = lf.filter(pl.col('WEEK_NUM').is_between(*VALID_WEEKS)).select(cols).collect(engine='streaming')
print(f'Train {CH_WEEKS}: {tr.height:,} | dr {tr["target"].mean():.4f} || Valid: {va.height:,} | dr {va["target"].mean():.4f}')

def to_woe(df, feats):
    out = np.empty((df.height, len(feats)), dtype=np.float64)
    for j, c in enumerate(feats):
        s = df[c]
        x = s.to_numpy() if s.dtype == pl.String else s.cast(pl.Float64).to_numpy()
        out[:, j] = binB[c].transform(x, metric='woe', metric_missing='empirical')
    return out

t0 = time.time()
Xtr = to_woe(tr, feat62); Xva = to_woe(va, feat62)
y_tr, y_va = tr['target'].to_numpy(), va['target'].to_numpy()
w_va = va['WEEK_NUM'].to_numpy()
print(f'WOE xong: {time.time()-t0:.0f}s')

# ---- 1. Kiểm chứng: model 62 đã lưu ----
p62 = logit62['model'].predict_proba(Xva)[:, 1]
m62 = gini_stability(w_va, y_va, p62)
print(f"\n[62 đã lưu] Valid {m62['gini_overall']:.4f} (notebook 06: 0.6175) | stab {m62['stability']:.4f}")

# ---- 2. Champion 61 (bỏ giới tính) ----
idx61 = [feat62.index(f) for f in feat61]
m61 = LogisticRegression(max_iter=2000).fit(Xtr[:, idx61], y_tr)
p61_tr = m61.predict_proba(Xtr[:, idx61])[:, 1]
p61 = m61.predict_proba(Xva[:, idx61])[:, 1]
r61 = gini_stability(w_va, y_va, p61)
gini_tr = float(2 * roc_auc_score(y_tr, p61_tr) - 1)
coef = pl.DataFrame({'feature': feat61, 'coef': m61.coef_[0]}).sort('coef')
print(f"[61 mới]    Train {gini_tr:.4f} | Valid {r61['gini_overall']:.4f} (notebook 06: 0.6067) | "
      f"gap {gini_tr - r61['gini_overall']:.4f} | stab {r61['stability']:.4f} "
      f"(mean {r61['mean_gini']:.4f}, a {r61['slope_a']:+.5f}, std_res {r61['std_resid']:.4f})")
print('Gini theo tuần:', r61['weekly'])
print(f"Hệ số dương (sai dấu kỳ vọng): {(coef['coef'] > 0).sum()} → {coef.filter(pl.col('coef') > 0)['feature'].to_list()}")

# ---- Lưu ----
CHAMPION = {'model': m61, 'feats': feat61, 'binning': {c: binB[c] for c in feat61},
            'woe_kwargs': {'metric': 'woe', 'metric_missing': 'empirical'},
            'train_weeks': CH_WEEKS, 'source': 'binning B (notebook 06) + Logistic refit, R7',
            'valid': {k: v for k, v in r61.items()}, 'gini_train': gini_tr}
joblib.dump(CHAMPION, MODELS / 'champion_final_61.joblib')
pl.DataFrame({'case_id': va['case_id'], 'WEEK_NUM': w_va, 'target': y_va, 'pred_champion': p61}
             ).write_parquet(REPORTS / 'champion_valid_preds.parquet')
coef.write_csv(REPORTS / 'champion_final_61_coef.csv')
with open(REPORTS / 'champion_final_61.json', 'w', encoding='utf-8') as f:
    json.dump({'feats': feat61, 'train_weeks': CH_WEEKS, 'gini_train': gini_tr, 'valid': r61,
               'check_full62_valid': m62['gini_overall']}, f, indent=2)
print('\nĐã lưu models/champion_final_61.joblib, reports/champion_valid_preds.parquet, champion_final_61_coef.csv, champion_final_61.json')

Cột tên có sex/gender: ['lastotherlnsexpense_631A', 'person_1__app__gender_992L', 'person_1__app__sex_738L']
Train (20, 43): 522,720 | dr 0.0307 || Valid: 224,421 | dr 0.0352
WOE xong: 7s

[62 đã lưu] Valid 0.6175 (notebook 06: 0.6175) | stab 0.4534
[61 mới]    Train 0.6249 | Valid 0.6067 (notebook 06: 0.6067) | gap 0.0182 | stab 0.5049 (mean 0.6043, a -0.00097, std_res 0.0280)
Gini theo tuần: {44: 0.6333, 45: 0.5866, 46: 0.5784, 47: 0.647, 48: 0.5769, 49: 0.6143, 50: 0.5695, 51: 0.6285}
Hệ số dương (sai dấu kỳ vọng): 0 → []

Đã lưu models/champion_final_61.joblib, reports/champion_valid_preds.parquet, champion_final_61_coef.csv, champion_final_61.json
